In [6]:
import pandas as pd
import random
import os

# Define our categories
CATEGORIES = ["sexism", "racism", "violence", "appearance", "ability", "non-offensive"]

def create_multi_label_tool(data_path, output_file, num_samples=100):
    """Tool to help manually label comments with multiple labels"""
    # Load the data
    df = pd.read_csv(data_path)
    
    # Check what column contains the comments
    print(f"Columns in your dataset: {df.columns.tolist()}")
    
    # Ask for the column name
    text_column = input(f"Which column contains the comments? Enter a name from above: ")
    
    if text_column not in df.columns:
        print(f"Error: Column '{text_column}' not found.")
        return
    
    # Select random samples
    if len(df) > num_samples:
        sample_df = df.sample(num_samples, random_state=42)
    else:
        sample_df = df
    
    # Create a new dataframe for labels
    labeled_df = pd.DataFrame({
        'comment': sample_df[text_column]
    })
    
    # Add columns for each category (1 for selected, 0 for not selected)
    for category in CATEGORIES:
        labeled_df[category] = 0
    
    # Add a column to track if this comment has been labeled
    labeled_df['is_labeled'] = False
    
    # Check if output file exists
    if os.path.exists(output_file):
        existing_df = pd.read_csv(output_file)
        print(f"Found existing labeled data with {len(existing_df)} entries.")
        
        # Make sure all category columns exist in the existing data
        for category in CATEGORIES:
            if category not in existing_df.columns:
                existing_df[category] = 0
        
        # Add 'is_labeled' column if it doesn't exist
        if 'is_labeled' not in existing_df.columns:
            existing_df['is_labeled'] = existing_df[CATEGORIES].sum(axis=1) > 0
        
        # Add only new comments
        existing_comments = set(existing_df['comment'].tolist())
        new_comments = [comment for comment in labeled_df['comment'] if comment not in existing_comments]
        
        if new_comments:
            new_df = pd.DataFrame({'comment': new_comments})
            for category in CATEGORIES:
                new_df[category] = 0
            new_df['is_labeled'] = False
            labeled_df = pd.concat([existing_df, new_df], ignore_index=True)
        else:
            labeled_df = existing_df
            print("No new comments to label.")
    
    # Interactive labeling
    for idx, row in labeled_df.iterrows():
        # Skip if already labeled
        if row['is_labeled']:
            continue
        
        # Display the comment
        print("\n" + "="*80)
        print(f"Comment {idx+1}/{len(labeled_df)}:")
        print(row['comment'])
        print("="*80)
        
        # Show category options
        print("\nCategories (select multiple by entering numbers separated by spaces):")
        for i, category in enumerate(CATEGORIES):
            print(f"{i+1}. {category}")
        
        # Get labels
        while True:
            choice = input("\nEnter category numbers (or 's' to skip, 'q' to quit): ")
            
            if choice.lower() == 'q':
                labeled_df.to_csv(output_file, index=False)
                print(f"Progress saved to {output_file}.")
                print(f"Labeled {labeled_df['is_labeled'].sum()} comments.")
                return
            
            if choice.lower() == 's':
                break
            
            try:
                # Parse multiple selections
                selections = [int(num) for num in choice.split()]
                valid_selections = all(1 <= num <= len(CATEGORIES) for num in selections)
                
                if valid_selections:
                    # Reset all categories for this comment
                    for category in CATEGORIES:
                        labeled_df.at[idx, category] = 0
                    
                    # Set the selected categories
                    for num in selections:
                        category = CATEGORIES[num-1]
                        labeled_df.at[idx, category] = 1
                    
                    # Mark as labeled
                    labeled_df.at[idx, 'is_labeled'] = True
                    break
                else:
                    print(f"Invalid choice. Please enter numbers between 1 and {len(CATEGORIES)}.")
            except ValueError:
                print("Please enter numbers separated by spaces, 's', or 'q'.")
        
        # Save progress after each label
        labeled_df.to_csv(output_file, index=False)
    
    # Final save
    labeled_df.to_csv(output_file, index=False)
    print(f"Labeling complete! Saved to {output_file}")
    print(f"Labeled {labeled_df['is_labeled'].sum()} comments.")
    
    # Ask if user wants to merge labels back to original dataset
    merge_choice = input("\nDo you want to merge the labels back to the original dataset? (y/n): ")
    if merge_choice.lower() == 'y':
        merge_labels_with_original(df, labeled_df, text_column, data_path)

def merge_labels_with_original(original_df, labeled_df, text_column, original_path):
    """Merge the labeled data back into the original dataset"""
    # Create output filename
    base_name = os.path.splitext(original_path)[0]
    merged_path = f"{base_name}_with_labels.csv"
    
    print("\nMerging labeled data with original dataset...")
    
    # Create a mapping of comments to their labels
    label_dict = {}
    for _, row in labeled_df.iterrows():
        if row['is_labeled']:
            label_dict[row['comment']] = {cat: row[cat] for cat in CATEGORIES}
    
    # Add label columns to original dataframe
    for category in CATEGORIES:
        original_df[category] = 0
    
    # Fill in labels where they exist
    labeled_count = 0
    for idx, row in original_df.iterrows():
        comment = row[text_column]
        if comment in label_dict:
            for category in CATEGORIES:
                original_df.at[idx, category] = label_dict[comment][category]
            labeled_count += 1
    
    # Save the merged dataset
    original_df.to_csv(merged_path, index=False)
    print(f"\nMerged dataset saved to {merged_path}")
    print(f"Added labels for {labeled_count} out of {len(original_df)} comments.")

if __name__ == "__main__":
    data_path = input("Enter the path to your dataset CSV file: ")
    output_file = input("Enter the path for saving labeled data (e.g., labeled_data.csv): ")
    num_samples = int(input("How many comments do you want to label? "))
    
    create_multi_label_tool(data_path, output_file, num_samples)

Columns in your dataset: ['Unnamed: 0', 'name', 'demographic', 'video_id', 'comment', 'author', 'like_count', 'published_at']

Comment 1/75:
Ronaldo is Tough, not like messy

Categories (select multiple by entering numbers separated by spaces):
1. sexism
2. racism
3. violence
4. appearance
5. ability
6. non-offensive

Comment 2/75:
But both are still goats

Categories (select multiple by entering numbers separated by spaces):
1. sexism
2. racism
3. violence
4. appearance
5. ability
6. non-offensive

Comment 3/75:
BRUH THIS IS FAKE

Categories (select multiple by entering numbers separated by spaces):
1. sexism
2. racism
3. violence
4. appearance
5. ability
6. non-offensive

Comment 4/75:
Ewww

Categories (select multiple by entering numbers separated by spaces):
1. sexism
2. racism
3. violence
4. appearance
5. ability
6. non-offensive

Comment 5/75:
A kid will fight over any thing

Categories (select multiple by entering numbers separated by spaces):
1. sexism
2. racism
3. violence
4. 

In [1]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

print("Bibliotheken erfolgreich importiert.")

Bibliotheken erfolgreich importiert.


In [7]:
# --- Konfiguration ---
# Pfad zu Ihrer CSV-Datei mit den 72 Features und dem Kommentartext
UNLABELED_CSV_PATH = '../Datasets/fe.csv'
# Pfad, unter dem die gelabelten Daten gespeichert werden sollen
LABELED_CSV_PATH = '../Datasets/labeled_comments.csv'
# Die Spalte, die den Text des Kommentars enthält
TEXT_COLUMN = 'Comment'
# Liste Ihrer Label-Kategorien
CATEGORIES = ['Sexism', 'Racism', 'Violence', 'Appearance', 'Ability', 'Non-offensive']
# Ziel: Wie oft soll jeder Feature-Wert pro Kategorie mindestens vorkommen?
MIN_OCCURRENCES = 3
# Anzahl der Kommentare, die in der Initialisierungsphase zufällig gelabelt werden
INITIAL_RANDOM_SAMPLES = 50

# --- Daten laden ---
try:
    df_unlabeled = pd.read_csv(UNLABELED_CSV_PATH)
    print(f"{len(df_unlabeled)} ungelabelte Kommentare geladen.")
except FileNotFoundError:
    print(f"Fehler: Die Datei '{UNLABELED_CSV_PATH}' wurde nicht gefunden.")
    # Erstellen Sie hier einen Dummy-DataFrame, um den Rest des Codes ausführen zu können
    df_unlabeled = pd.DataFrame({
        TEXT_COLUMN: [f"Das ist ein Testkommentar {i}" for i in range(100)],
        'feature_1': np.random.randint(0, 2, 100),
        'feature_2': np.random.choice(['A', 'B', 'C'], 100)
    })
    print("Ein Dummy-DataFrame wurde für Demonstrationszwecke erstellt.")

# Feature-Spalten identifizieren (alle außer der Textspalte)
FEATURE_COLUMNS = [col for col in df_unlabeled.columns if col != TEXT_COLUMN]

# DataFrame für gelabelte Daten vorbereiten
if os.path.exists(LABELED_CSV_PATH):
    df_labeled = pd.read_csv(LABELED_CSV_PATH)
    print(f"{len(df_labeled)} bereits gelabelte Kommentare geladen.")
else:
    df_labeled = pd.DataFrame(columns=df_unlabeled.columns.tolist() + ['label'])
    print("Keine bestehende Labeling-Datei gefunden. Eine neue wird erstellt.")

# Bereits gelabelte Kommentare aus dem ungelabelten Set entfernen
if not df_labeled.empty:
    df_unlabeled = df_unlabeled.loc[~df_unlabeled[TEXT_COLUMN].isin(df_labeled[TEXT_COLUMN])]

print(f"Verbleibende ungelabelte Kommentare: {len(df_unlabeled)}")

178846 ungelabelte Kommentare geladen.
382 bereits gelabelte Kommentare geladen.
Verbleibende ungelabelte Kommentare: 177025


In [8]:
def get_coverage_matrix(df_labeled, features, categories):
    """
    Erstellt eine Matrix, die zählt, wie oft jeder Feature-Wert pro Kategorie vorkommt.
    Angepasst für Multi-Label-Daten (binäre Spalten für Kategorien).
    """
    if df_labeled.empty or df_labeled[categories].sum().sum() == 0:
        return None # Keine gelabelten Daten vorhanden

    coverage_per_feature = {}
    for feature in features:
        # Initialisiere eine leere DataFrame für die Abdeckung dieses Features
        feature_values = df_labeled[feature].unique()
        coverage_df = pd.DataFrame(0, index=feature_values, columns=categories)
        
        for category in categories:
            # Filtere die Zeilen, in denen dieses Label gesetzt ist
            labeled_with_cat = df_labeled[df_labeled[category] == 1]
            if not labeled_with_cat.empty:
                # Zähle die Vorkommen der Feature-Werte
                counts = labeled_with_cat[feature].value_counts()
                # Update die Abdeckungs-DataFrame
                coverage_df[category].update(counts)
        
        coverage_per_feature[feature] = coverage_df
        
    return coverage_per_feature

def calculate_priority_score(row, coverage, features, categories, min_occurrences):
    """
    Berechnet einen Prioritätsscore für eine einzelne Datenzeile (Kommentar).
    Die Logik bleibt gleich, nutzt aber die neue Coverage-Matrix-Struktur.
    """
    score = 0
    if coverage is None:
        return np.random.rand() # Zufällige Priorität in der Initialisierungsphase

    for feature in features:
        value = row[feature]
        for cat in categories:
            count = 0
            # Sicherstellen, dass der Feature-Wert und die Kategorie in der Matrix existieren
            if feature in coverage and value in coverage[feature].index:
                count = coverage[feature].loc[value, cat]
            
            if count < min_occurrences:
                score += (min_occurrences - count)
    return score

def get_next_comment_index(df_unlabeled, df_labeled, features, categories, min_occurrences, initial_samples):
    """
    Bestimmt den Index des nächsten zu labelnden Kommentars. (Funktion ist weitgehend unverändert)
    """
    if len(df_labeled) < initial_samples or df_unlabeled.empty:
        if df_unlabeled.empty:
            return None
        return np.random.choice(df_unlabeled.index)

    coverage = get_coverage_matrix(df_labeled, features, categories)
    
    df_unlabeled_copy = df_unlabeled.copy()
    
    sample_size = min(1000, len(df_unlabeled_copy))
    sample_indices = np.random.choice(df_unlabeled_copy.index, sample_size, replace=False)
    sample_df = df_unlabeled_copy.loc[sample_indices]

    scores = sample_df.apply(
        lambda row: calculate_priority_score(row, coverage, features, categories, min_occurrences),
        axis=1
    )
    
    return scores.idxmax()

In [10]:
# --- Widgets für die UI erstellen (Multi-Label Version) ---
comment_text_area = widgets.Textarea(
    value='',
    placeholder='Hier erscheint der Kommentartext...',
    description='Kommentar:',
    disabled=True,
    layout={'height': '150px', 'width': '95%'}
)

progress_label = widgets.Label(value=f"Gelabelt: {len(df_labeled)} / {len(df_labeled) + len(df_unlabeled)}")
status_label = widgets.Label(value="Status: Bereit")

# Checkboxes für jede Kategorie
checkboxes = [widgets.Checkbox(value=False, description=cat) for cat in CATEGORIES]
checkbox_container = widgets.VBox(children=checkboxes, layout=widgets.Layout(padding='10px'))

# Submit-Button
submit_button = widgets.Button(description="Label speichern & Nächster", button_style='primary', layout=widgets.Layout(width='200px'))
save_and_exit_button = widgets.Button(description="Speichern & Beenden", button_style='success', layout=widgets.Layout(width='200px'))

# UI-Container
ui_container = widgets.VBox([
    progress_label,
    comment_text_area,
    widgets.Label("Wählen Sie alle zutreffenden Kategorien aus:"),
    checkbox_container,
    submit_button,
    status_label,
])

# Globale Variable für den aktuellen Index
current_index = None

def display_next_comment():
    """Holt den nächsten Kommentar und zeigt ihn in der UI an."""
    global current_index, df_unlabeled

    # Checkboxes zurücksetzen
    for cb in checkboxes:
        cb.value = False
    
    if df_unlabeled.empty:
        comment_text_area.value = "Herzlichen Glückwunsch! Alle Kommentare wurden gelabelt."
        status_label.value = "Status: Fertig"
        submit_button.disabled = True
        return

    index = get_next_comment_index(df_unlabeled, df_labeled, FEATURE_COLUMNS, CATEGORIES, MIN_OCCURRENCES, INITIAL_RANDOM_SAMPLES)
    current_index = index
    
    comment_text_area.value = df_unlabeled.loc[index, TEXT_COLUMN]
    progress_label.value = f"Gelabelt: {len(df_labeled)} / {len(df_labeled) + len(df_unlabeled)}"
    phase = "Initialisierung (Zufällig)" if len(df_labeled) < INITIAL_RANDOM_SAMPLES else "Geführtes Labeling (Priorisiert)"
    status_label.value = f"Status: {phase}"

def on_submit_button_clicked(b):
    """Event-Handler für den Submit-Button."""
    global df_labeled, df_unlabeled, current_index

    if current_index is None: return

    # Die Feature-Daten des Kommentars kopieren
    labeled_row_features = df_unlabeled.loc[[current_index]].copy()
    
    # Die Labels aus den Checkboxes auslesen und als neue Spalten hinzufügen
    for cb in checkboxes:
        labeled_row_features[cb.description] = 1 if cb.value else 0
    
    # Zu den gelabelten Daten hinzufügen
    df_labeled = pd.concat([df_labeled, labeled_row_features], ignore_index=True)
    
    # Aus den ungelabelten Daten entfernen
    df_unlabeled = df_unlabeled.drop(current_index)
    
    # Automatisch speichern
    if len(df_labeled) % 10 == 0:
        df_labeled.to_csv(LABELED_CSV_PATH, index=False)
        status_label.value = f"Fortschritt wurde in '{LABELED_CSV_PATH}' gespeichert."

    display_next_comment()

def on_save_and_exit_clicked(b):
    """Event-Handler für den Speichern & Beenden-Button."""
    df_labeled.to_csv(LABELED_CSV_PATH, index=False)
    status_label.value = f"Final gespeichert in '{LABELED_CSV_PATH}'. Beendet."
    submit_button.disabled = True
    save_and_exit_button.disabled = True

# Event-Handler an die Buttons binden
submit_button.on_click(on_submit_button_clicked)
save_and_exit_button.on_click(on_save_and_exit_clicked)

# --- Start der Anwendung ---
clear_output()
display(ui_container, save_and_exit_button)
display_next_comment()

Button(button_style='success', description='Speichern & Beenden', layout=Layout(width='200px'), style=ButtonSt…

C:\Users\Daniel\AppData\Local\Temp\ipykernel_6752\3803216414.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  coverage_df[category].update(counts)


In [5]:
import pandas as pd
import numpy as np

# --- Konfiguration (Bitte stellen Sie sicher, dass diese mit Ihrem Labeling-Skript übereinstimmt) ---
LABELED_CSV_PATH = '../Datasets/labeled_comments.csv'
TEXT_COLUMN = 'Comment' # Wird hier nur zum Aussortieren benötigt
CATEGORIES = ['Sexism', 'Racism', 'Violence', 'Appearance', 'Ability', 'Non-offensive']
MIN_OCCURRENCES = 3

# --- KORRIGIERTE Funktion ---
def get_coverage_matrix(df_labeled, features, categories):
    """
    Erstellt eine Matrix, die zählt, wie oft jeder Feature-Wert pro Kategorie vorkommt.
    Angepasst für Multi-Label-Daten (binäre Spalten für Kategorien).
    """
    if df_labeled.empty or df_labeled[categories].sum().sum() == 0:
        return None # Keine gelabelten Daten vorhanden

    coverage_per_feature = {}
    for feature in features:
        # Extrahiere einzigartige Werte aus der *gesamten* Spalte, um sicherzustellen, dass alle erfasst werden
        feature_values = df_labeled[feature].unique()
        coverage_df = pd.DataFrame(0, index=feature_values, columns=categories)

        for category in categories:
            # Filtere die Zeilen, in denen dieses Label gesetzt ist (Wert ist 1)
            labeled_with_cat = df_labeled[df_labeled[category] == 1]
            if not labeled_with_cat.empty:
                # Zähle die Vorkommen der Feature-Werte in den gefilterten Daten
                counts = labeled_with_cat[feature].value_counts()
                
                # --- KORRIGIERTE STELLE ---
                # Verwende .update(), da dies robuster gegen Index-Fehlinterpretationen ist.
                # Es aktualisiert die Werte in der 'category'-Spalte basierend auf dem Index von 'counts'.
                coverage_df[category].update(counts)

        coverage_per_feature[feature] = coverage_df

    return coverage_per_feature


# --- Haupt-Analyseskript (Rest ist unverändert) ---
print("Starte Analyse der Repräsentation...")

try:
    df_labeled = pd.read_csv(LABELED_CSV_PATH)
    df_labeled = df_labeled.drop('Unnamed: 0', axis=1)
    print(f"'{LABELED_CSV_PATH}' erfolgreich geladen. {len(df_labeled)} gelabelte Kommentare gefunden.")
except FileNotFoundError:
    print(f"FEHLER: Die Datei '{LABELED_CSV_PATH}' wurde nicht gefunden.")
    print("Bitte führen Sie zuerst das Labeling-Skript aus, um einige Kommentare zu labeln.")
    # Beendet das Skript, wenn die Datei nicht existiert
    exit()

if df_labeled.empty:
    print("Die Label-Datei ist leer. Es gibt noch keine Repräsentation zu analysieren.")
    exit()

# Feature-Spalten dynamisch identifizieren
feature_columns = [col for col in df_labeled.columns if col not in CATEGORIES + [TEXT_COLUMN]]

print(f"\nAnalysiere {len(feature_columns)} Features für {len(CATEGORIES)} Kategorien...")
print("-" * 60)

# Abdeckungsmatrix berechnen
coverage_data = get_coverage_matrix(df_labeled, feature_columns, CATEGORIES)

# --- 1. Zusammenfassung der unterrepräsentierten Bereiche ---
print(f"\nANALYSE: BEREICHE MIT WENIGER ALS {MIN_OCCURRENCES} LABELS\n")
underrepresented_found = False
if coverage_data:
    for feature, coverage_df in coverage_data.items():
        # Finde Zellen, in denen der Zählwert > 0, aber < MIN_OCCURRENCES ist
        underrepresented_df = coverage_df[(coverage_df > 0) & (coverage_df < MIN_OCCURRENCES)]
        # Entferne Zeilen und Spalten, die nur leere Werte (NaN/0) enthalten, für eine saubere Ausgabe
        underrepresented_df = underrepresented_df.dropna(how='all').dropna(how='all', axis=1)

        if not underrepresented_df.empty:
            underrepresented_found = True
            print(f"▶ Feature: '{feature}'")
            # Iteriere durch die Ergebnisse für eine saubere Ausgabe
            for value, row in underrepresented_df.iterrows():
                for category, count in row.dropna().items():
                    print(f"  - Wert '{value}' | Kategorie '{category}': Nur {int(count)} von {MIN_OCCURRENCES} Labels vorhanden.")
            print() # Leerzeile für Lesbarkeit
else:
     print("Konnte keine Abdeckungsdaten berechnen (möglicherweise noch keine Labels gesetzt?).")

if not underrepresented_found:
    print("✅ GLÜCKWUNSCH! Alle bisher gelabelten Feature-Werte sind in allen relevanten Kategorien ausreichend repräsentiert.")
    print("Das bedeutet, für jede Kombination aus Feature-Wert und Kategorie, die Sie bisher gelabelt haben, wurde das Ziel von 3 erreicht.")

print("-" * 60)

# --- 2. Optionale: Vollständige Repräsentation anzeigen ---
show_full_matrix = input("Möchten Sie die komplette Abdeckungsmatrix für alle Features anzeigen? (j/n): ")
if show_full_matrix.lower() == 'j':
    print("\n--- Vollständige Abdeckungsmatrix ---\n")
    if coverage_data:
        for feature, coverage_df in coverage_data.items():
            print(f"Feature: '{feature}'")
            # Zeige die volle Matrix ohne Zeilenlimit an
            with pd.option_context('display.max_rows', None):
                print(coverage_df)
            print("-" * 30)
    else:
        # Dieser Fall sollte bereits oben abgefangen sein, aber zur Sicherheit
        print("Keine Daten zum Anzeigen vorhanden.")

print("\nAnalyse beendet.")

Starte Analyse der Repräsentation...
'../Datasets/labeled_comments.csv' erfolgreich geladen. 382 gelabelte Kommentare gefunden.

Analysiere 75 Features für 6 Kategorien...
------------------------------------------------------------


C:\Users\Daniel\AppData\Local\Temp\ipykernel_6752\1010072588.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  coverage_df[category].update(counts)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_6752\1010072588.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when


ANALYSE: BEREICHE MIT WENIGER ALS 3 LABELS

▶ Feature: 'Video Popularity'
  - Wert '1' | Kategorie 'Sexism': Nur 1 von 3 Labels vorhanden.
  - Wert '1' | Kategorie 'Racism': Nur 1 von 3 Labels vorhanden.
  - Wert '1' | Kategorie 'Violence': Nur 1 von 3 Labels vorhanden.
  - Wert '0' | Kategorie 'Sexism': Nur 2 von 3 Labels vorhanden.
  - Wert '0' | Kategorie 'Violence': Nur 2 von 3 Labels vorhanden.
  - Wert '0' | Kategorie 'Appearance': Nur 1 von 3 Labels vorhanden.
  - Wert '2' | Kategorie 'Sexism': Nur 2 von 3 Labels vorhanden.
  - Wert '2' | Kategorie 'Violence': Nur 1 von 3 Labels vorhanden.
  - Wert '2' | Kategorie 'Appearance': Nur 1 von 3 Labels vorhanden.
  - Wert '2' | Kategorie 'Ability': Nur 2 von 3 Labels vorhanden.

▶ Feature: 'Comment Popularity'
  - Wert '0' | Kategorie 'Violence': Nur 1 von 3 Labels vorhanden.
  - Wert '0' | Kategorie 'Appearance': Nur 2 von 3 Labels vorhanden.
  - Wert '0' | Kategorie 'Ability': Nur 1 von 3 Labels vorhanden.
  - Wert '1' | Kategorie 